In [ ]:
import pandas as pd

# Import data
df_master=pd.read_csv(master_raw.csv)
df_ref=pd.read_csv(reference_raw.csv)

### 1️. Transform MASTER file

# Columns to keep and rename
master_cols = {
    "N-NUMBER": "n_number",
    "MFR MDL CODE": "mfr_mdl_code",
    "YEAR MFR": "year_manufactured",
    "TYPE AIRCRAFT": "aircraft_type_code",
    "TYPE ENGINE": "engine_type_code",
    "CERTIFICATION": "certification_code",
    "STATUS CODE": "registration_status",
    "CERT ISSUE DATE": "cert_issue_date",
    "EXPIRATION DATE": "expiration_date",
    "STATE": "owner_state",
    "COUNTRY": "owner_country"
}

df_master = df_master[list(master_cols.keys())].rename(columns=master_cols)

# Clean status and filter active
df_master["registration_status"] = df_master["registration_status"].astype(str).str.strip().str.upper()
df_master = df_master[df_master["registration_status"] == "V"]

# Convert types
df_master["year_manufactured"] = pd.to_numeric(df_master["year_manufactured"], errors="coerce")
df_master[["cert_issue_date", "expiration_date"]] = df_master[["cert_issue_date", "expiration_date"]].apply(pd.to_datetime, errors="coerce")

# Handle missing values
df_master["owner_state"] = df_master["owner_state"].fillna("UNKNOWN")
df_master["owner_country"] = df_master["owner_country"].fillna("UNKNOWN")
df_master["certification_code"] = df_master["certification_code"].astype(str).str.strip().replace({"": "UNKNOWN", "error": "UNKNOWN"})

# Normalize text columns
text_cols = ["n_number", "mfr_mdl_code", "aircraft_type_code", "engine_type_code", "certification_code"]
for col in text_cols:
    df_master[col] = df_master[col].astype(str).str.strip().replace({"": "UNKNOWN", "error": "UNKNOWN"})

# Remove duplicates
df_master = df_master.drop_duplicates(subset="n_number")

# Save clean MASTER
df_master.to_csv("aircraft_master_clean.csv", index=False)


### 2️. Transform ACFTREF file

# Clean column names
df_ref.columns = df_ref.columns.str.strip()

# Columns to keep and rename
ref_cols = {
    "CODE": "mfr_mdl_code",
    "MFR": "manufacturer",
    "MODEL": "model",
    "TYPE-ACFT": "aircraft_category",
    "TYPE-ENG": "engine_type",
    "NO-ENG": "engine_count",
    "AC-WEIGHT": "weight_class"
}

df_ref = df_ref[list(ref_cols.keys())].rename(columns=ref_cols)

# Clean categorical columns
for col in ["aircraft_category", "engine_type", "weight_class", "manufacturer", "model"]:
    df_ref[col] = df_ref[col].astype(str).str.strip().replace({"": "UNKNOWN", "error": "UNKNOWN"})

# Convert types
df_ref["mfr_mdl_code"] = df_ref["mfr_mdl_code"].astype(str)
df_ref["aircraft_category"] = df_ref["aircraft_category"].astype("category")
df_ref["engine_type"] = df_ref["engine_type"].astype("category")
df_ref["weight_class"] = df_ref["weight_class"].astype("category")
df_ref["engine_count"] = pd.to_numeric(df_ref["engine_count"], errors="coerce")

# Drop rows with missing primary key
df_ref = df_ref[df_ref["mfr_mdl_code"] != ""]

# Save clean ACFTREF
df_ref.to_csv("aircraft_reference_clean.csv", index=False)


### 3️. Join MASTER + ACFTREF to create analytics dataset

df_analytics = df_master.merge(df_ref, on="mfr_mdl_code", how="left")

# Fill missing with "UNKNOWN" for categorical columns
cat_fill_cols = ["manufacturer", "model", "aircraft_category", "engine_type", "weight_class"]
for col in cat_fill_cols:
    if df_analytics[col].dtype.name == "category":
        df_analytics[col] = df_analytics[col].cat.add_categories("UNKNOWN")
    df_analytics[col] = df_analytics[col].fillna("UNKNOWN")

# Save final analytics-ready table
df_analytics.to_csv("faa_analytics_dataset.csv", index=False)

